In [79]:
import duckdb
con = duckdb.connect("data/flights.duckdb")

In [80]:
con.execute('''DESCRIBE fact_flights''').df()

,column_name,column_type,null,key,default,extra
0,flight_date,DATE,YES,None,None,None
1,carrier_code,VARCHAR,YES,None,None,None
2,origin_airport,VARCHAR,YES,None,None,None
3,dest_airport,VARCHAR,YES,None,None,None
4,dep_hour,INTEGER,YES,None,None,None
5,dep_delay,DOUBLE,YES,None,None,None
6,arr_delay,DOUBLE,YES,None,None,None
7,delayed,INTEGER,YES,None,None,None
8,cancelled,INTEGER,YES,None,None,None
9,diverted,INTEGER,YES,None,None,None


In [81]:
model_df = con.execute('''SELECT flight_date, carrier_code, origin_airport, dest_airport, dep_hour, distance, day_of_week, is_weekend, is_holiday, delayed FROM fact_flights
 LEFT JOIN dim_date ON flight_date = date WHERE delayed IS NOT NULL ORDER BY flight_date, carrier_code, origin_airport, dest_airport, dep_hour, distance, day_of_week, is_weekend, is_holiday, delayed ''').df()
display(model_df)
assert(0.2 < model_df["delayed"].mean() < 0.25)
print("Average Delay Rate:", model_df["delayed"].mean())
model_df.isna().sum() # We have no nulls


,flight_date,carrier_code,origin_airport,dest_airport,dep_hour,distance,day_of_week,is_weekend,is_holiday,delayed
0,2025-08-01,AA,ABQ,DFW,5,569.0,5,False,False,0
1,2025-08-01,AA,ABQ,DFW,6,569.0,5,False,False,0
2,2025-08-01,AA,ABQ,DFW,8,569.0,5,False,False,1
3,2025-08-01,AA,ABQ,DFW,10,569.0,5,False,False,0
4,2025-08-01,AA,ABQ,DFW,17,569.0,5,False,False,1
...,...,...,...,...,...,...,...,...,...,...
6895505,2026-07-31,YX,TVC,EWR,13,644.0,5,False,False,0
6895506,2026-07-31,YX,TVC,LGA,17,655.0,5,False,False,0
6895507,2026-07-31,YX,XNA,LGA,6,1147.0,5,False,False,0
6895508,2026-07-31,YX,XNA,LGA,9,1147.0,5,False,False,0


Average Delay Rate: 0.2168239912638804


flight_date       0
carrier_code      0
origin_airport    0
dest_airport      0
dep_hour          0
distance          0
day_of_week       0
is_weekend        0
is_holiday        0
delayed           0
dtype: int64

Row count = 6895510. delay_rate = 0.2168. 0 NULLs in table.

In [82]:
import datetime

In [83]:
training_data = model_df[model_df["flight_date"] < datetime.datetime(2026, 6, 1)]
print(training_data.shape)
print("Average delayed rate of training data:", training_data["delayed"].mean())

testing_data = model_df[model_df["flight_date"] >= datetime.datetime(2026, 6, 1)]
print(testing_data.shape)
print("Average delayed rate of testing data:", testing_data["delayed"].mean())

assert(len(testing_data) + len(training_data) == len(model_df))


(5688277, 10)
Average delayed rate of training data: 0.20697515258135285
(1207233, 10)
Average delayed rate of testing data: 0.2632300475550287


Training data Rows  X Columns: (5688277, 10) \
Average delayed rate of training data: 0.20697515258135285 \
Testing data Rows  X Columns: (1207233, 10) \
Average delayed rate of testing data: 0.2632300475550287

# First Model

set up X and y

In [84]:
X_train = training_data.loc[:, ['carrier_code', 'origin_airport', 'dest_airport', 'dep_hour', 'distance', 'day_of_week', 'is_holiday']]
y_train = training_data["delayed"]
assert(X_train.size/len(X_train) + y_train.size/len(y_train) == 8)

Sort Columns into Groups

In [85]:
categorical_cols = ['carrier_code', 'origin_airport', 'dest_airport', 'dep_hour', 'day_of_week']
numeric_cols = ['distance']
passthrough_cols = ['is_holiday']

In [86]:
from sklearn.preprocessing import OneHotEncoder

In [87]:
encoder = OneHotEncoder()

In [88]:
sample = X_train[["carrier_code"]].sample(5, random_state=42) #Randomly samples 5 to avoid the same airline appearing, then random_state=42 makes it the same random sample every time
print(sample)
encoder.fit(sample)
encoder.transform(sample).toarray()
encoder.categories_

        carrier_code
106926            DL
1654696           UA
1186101           WN
3289332           NK
1244724           WN


[array(['DL', 'NK', 'UA', 'WN'], dtype=object)]

In [89]:
from sklearn.preprocessing import StandardScaler

In [90]:
scaler = StandardScaler()

# Scaling distance:
 Distances are in the thousands, but our other columns are just 0 or 1. The scaler shrinks distance onto a similar scale by turning each value into a z-score: how far it is from the average, measured in standard deviations. It first fits (learns the average and spread) and then transforms (converts each distance using those numbers).

In [91]:
sample2 = X_train[["distance"]].sample(5, random_state=42) # random 5 distances
scaler.fit(sample2)
scaled = scaler.transform(sample2)
print(scaled)
sample2_mean = sample2.mean()
sample2_sd = sample2.std()
print("mean:", scaler.mean_, "scale:", scaler.scale_, "sample2 SD:", sample2_sd)
chosen_distance_z_score = (sample2.iloc[0]-sample2_mean)/sample2_sd
print("Chosen Distance Z-Score:", chosen_distance_z_score, "Distance Z-score scaled from sample2:", scaled[0])

[[-0.9139605 ]
 [ 1.86968801]
 [-0.74689245]
 [ 0.10687441]
 [-0.31570947]]
mean: [896.] scale: [814.03955678] sample2 SD: distance    910.123893
dtype: float64
Chosen Distance Z-Score: distance   -0.817471
dtype: float64 Distance Z-score scaled from sample2: [-0.9139605]


The pandas built in `.std()` uses the sample standard deviation formula: $$s = \sqrt{\frac{\sum_{i=1}^{n} (x_i - \bar{x})^2}{n - 1}}$$ while the scaler uses the population standard deviation: $$\sigma = \sqrt{\frac{\sum_{i=1}^{N} (x_i - \mu)^2}{N}}$$ 
#### What does this mean?
    Although it makes a noticable difference in the Z-score here we will not have to worry about it in the future. It is noticable here because with only 5 values the sample sd formula divides by 4 while the population sd formula divides by 5. When we run our real training data we will have about 5,000,000 flights so dividing by 4,999,999 makes a very small difference in comparison to dividing by 5,000,000.

## Bringing Everything Back Together in a DataFrame

In [92]:
from sklearn.compose import ColumnTransformer

In [93]:
transformer = ColumnTransformer([("categorical columns", OneHotEncoder(handle_unknown="ignore"), categorical_cols), ("numeric columns", StandardScaler(), numeric_cols), ("passthrough columns", "passthrough", passthrough_cols)])
#transformer turns flights into numbers. It leanrs categories, mean, SD
X_small = X_train.sample(500000, random_state=42)
transformer.fit(X_small)
print(X_small.shape)
print(X_small[categorical_cols].nunique())
transformed = transformer.transform(X_small) #holds the output of the data after transformation
print(transformed.shape)


(500000, 7)
carrier_code       14
origin_airport    354
dest_airport      353
dep_hour           24
day_of_week         7
dtype: int64
(500000, 754)


# Creating the model

The goal is to predict whether or not a flight will be delayed (arrival more than 15 minutes late) using only information before the departure. \
The model will assign each flight with a probability score with how likely this flight is to be in the delayed group (arrival > expected arrival + 15) \
We then see if the model ranks these flights scores well and if they match up with which ones were actually delayed.
\
### Appraoch
We are using logistic regression. It judges every single feature and determines if they have a positive or negative correlation with delay risk. \
### Preperation
The model only does math on numbers, which is why we used ColumnTransformer to convert the features. We converted the categorical columns to 1s and 0s and we turned the distance column to z-scores.
### Pipeline
Connects the transformer to the model such that the settings that we applied (1/0 columns, z-scores) are applied to the test data, and this will prevent Train-test contamination leakage.

## Training Data: Aug 2025 to May 2026; Test Data: June and July 2026 (untouched until the test protocol is fixed)




In [94]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [95]:
model = LogisticRegression()
print(model)

LogisticRegression()


### Pipeline
 Connecting transformer to model

In [96]:
pipe = Pipeline([("preperation", transformer), ("model", model)])
print(pipe)

Pipeline(steps=[('preperation',
                 ColumnTransformer(transformers=[('categorical columns',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['carrier_code',
                                                   'origin_airport',
                                                   'dest_airport', 'dep_hour',
                                                   'day_of_week']),
                                                 ('numeric columns',
                                                  StandardScaler(),
                                                  ['distance']),
                                                 ('passthrough columns',
                                                  'passthrough',
                                                  ['is_holiday'])])),
                ('model', LogisticRegression())])


In [105]:
y_small = y_train.loc[X_small.index]
print(y_small.shape)
assert(X_small.index.equals(y_small.index))
pipe.fit(X_small, y_small)

(500000,)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preperation', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int32](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['carrier_code','origin_airport','dest_airport',...,'distance', 'day_of_week','is_holiday']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical columns', ...), ('numeric columns', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'`

## Grading the Model Using Training Data
### Use Cross-Validaiton
 Use sample of flights from training data. \
 Split flights into 5 piles, Train on 4 piles then grade on 5th. \
 Repeat 5 times so every pile gets a turn as the grading pile.

#### AUC
 The grade that we use. Pick one delayed flitght and one on time one at random. \
 AUC is the chance that the model gave the delayed flight the higher risk score out of the two.


In [106]:
from sklearn.model_selection import cross_val_score

In [ ]:
scores = cross_val_score(pipe, X_small, y_small, cv=5, scoring="roc_auc")
print(f" 5 AUC scores: {scores}\n Mean Score : {scores.mean()}\n Standard Deviation of Scores: {scores.std()}")

5 AUC scores: [0.65414027 0.65097865 0.65061184 0.65446188 0.65215774]
 Mean Score : 0.652470075794064
 Standard Deviation of Scores: 0.0015831479417885905


If you pick one delayed and one on-time flight at random, the model gives the delayed one the higher risk score about 65% of the time. The 5 folds differ by only about 0.002, so the score is stable, not luck.